In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import time
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
import warnings
warnings.filterwarnings('ignore')

In [2]:
def selectkbest(indep_X, dep_Y, n):
    test = SelectKBest(score_func=chi2, k=n)
    fit1 = test.fit(indep_X, dep_Y)
    selectK_features = fit1.transform(indep_X)
    return selectK_features

def split_scaler(indep_X, dep_Y):
    X_train,X_test,y_train,y_test=train_test_split(indep_X, dep_Y, test_size= 0.25, random_state = 0)
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)
    
    return  X_train,X_test,y_train,y_test

def cm_prediction(classifier,X_test):
    
    y_pred= classifier.predict(X_test)

    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_test,y_pred)

    from sklearn.metrics import accuracy_score
    from sklearn.metrics import classification_report
    Accuracy=accuracy_score(y_test, y_pred)

    report = classification_report(y_test,y_pred)
    
    return classifier,Accuracy,report,X_test,y_test,cm

def logistic(X_train,y_train,X_test):
    
    classifier=LogisticRegression(max_iter=1000)
    classifier.fit(X_train,y_train)
    classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
    
    return classifier,Accuracy,report,X_test,y_test,cm


def random(X_train,y_train,X_test):
    
    classifier=RandomForestClassifier(n_estimators=100,criterion='entropy',random_state=0)
    classifier.fit(X_train,y_train)
    classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
    
    return classifier,Accuracy,report,X_test,y_test,cm

In [3]:
def selectk_classification(acclog, accrf):
    
    dataframe = pd.DataFrame(index=['chisquare'],columns=['Logistic','Random'])
    
    for number, idex in enumerate(dataframe.index):
        dataframe.loc[idex, 'Logistic'] = acclog[number]
        dataframe.loc[idex, 'Random'] = accrf[number]
    return dataframe

In [4]:
dataset=pd.read_csv('telecom_churn.csv',index_col=None)

In [5]:
df2 = dataset

In [6]:
df2.drop(['customer_id','pincode','date_of_registration','data_used','state','city','calls_made','sms_sent'], axis=1, inplace=True)

In [7]:
df2=pd.get_dummies(df2, drop_first=True, dtype=int)

In [8]:
df2

,age,num_dependents,estimated_salary,churn,telecom_partner_BSNL,telecom_partner_Reliance Jio,telecom_partner_Vodafone,gender_M
0,25,4,124962,0,0,1,0,0
1,55,2,130556,0,0,1,0,0
2,57,0,148828,1,0,0,1,0
3,46,1,38722,1,1,0,0,1
4,26,2,55098,0,1,0,0,0
...,...,...,...,...,...,...,...,...
243548,28,3,130580,0,0,0,0,0
243549,52,0,82393,0,0,1,0,0
243550,59,4,51298,0,0,1,0,1
243551,49,2,83981,0,1,0,0,1


In [9]:
indep_X=df2.drop('churn',axis=1)
dep_Y=df2['churn']

In [10]:
dep_Y

0         0
1         0
2         1
3         1
4         0
         ..
243548    0
243549    0
243550    0
243551    0
243552    0
Name: churn, Length: 243553, dtype: int64

In [23]:
kbest=selectkbest(indep_X,dep_Y,7)
acclog, accrf = [], []


In [24]:
kbest

array([[    25,      4, 124962, ...,      1,      0,      0],
       [    55,      2, 130556, ...,      1,      0,      0],
       [    57,      0, 148828, ...,      0,      1,      0],
       ...,
       [    59,      4,  51298, ...,      1,      0,      1],
       [    49,      2,  83981, ...,      0,      0,      1],
       [    37,      0, 144297, ...,      0,      0,      0]],
      shape=(243553, 7))

In [25]:
X_train,X_test,y_train,y_test=split_scaler(kbest,dep_Y)

classifier,Accuracy,report,X_test,y_test,cm=logistic(X_train,y_train,X_test)
acclog.append(Accuracy)

classifier,Accuracy,report,X_test,y_test,cm=random(X_train,y_train,X_test)
accrf.append(Accuracy)

result=selectk_classification(acclog,accrf)

In [14]:
result #3

,Logistic,Random
chisquare,0.79911,0.684179


In [18]:
result #5

,Logistic,Random
chisquare,0.79911,0.70011


In [22]:
result #6

,Logistic,Random
chisquare,0.79911,0.703838


In [26]:
result #7

,Logistic,Random
chisquare,0.79911,0.707648
